<a href="https://colab.research.google.com/github/Tristan-Toye/Information-retrieval-and-search-engines/blob/Development/IRSE_Project_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# H02C8b Information Retrieval and Search Engines: RAG Project

Welcome to the notebook companion for the IRSE project. You will find all starter code here. You are encouraged to use this code, as it has been confirmed to work for the RAG pipeline described in the assignment handout. However, you are certainly welcome to make any changes you see fit, provided that your code is written in Python and runs without issue.

**IMPORTANT**: Do not submit a notebook as your final solution. It will not be graded. Refer to assignment handout for more information about the submission format.

**IMPORTANT**: Be mindful of your runtime usage, if working in Colab. At the beginning of every session, navigate to the top menu bar in Colab and select **Runtime > Change runtime type > CPU (Python 3)**. This will ensure that your session runs on CPU and that you do not waste any GPU allocation for the day. GPUs are provided by Google on a limited daily basis, and access is given every 24 hours. It is best that you complete the TF-IDF/search component before loading models and running inference on the GPU runtime.


If you have any questions, feel free to email [Thomas](mailto:thomas.bauwens@kuleuven.be) or [Kushal](mailto:kushaljayesh.tatariya@kuleuven.be).

## RAG for recipe recommendation:

We will begin by installing the huggingface `datasets` library for easily loading our data.

https://stackoverflow.com/questions/3788870/how-to-check-if-a-word-is-an-english-word-with-python
https://github.com/slanglab/phrasemachine
https://www.nltk.org/api/nltk.tokenize.mwe.html-> multi word tags




"https://huggingface.co/collections/open-llm-leaderboard/open-llm-leaderboard-best-models-652d6c7965a4619fb5c27a03"
"https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct"
"https://huggingface.co/deepseek-ai/deepseek-coder-7b-instruct-v1.5"
"https://huggingface.co/deepseek-ai/deepseek-coder-6.7b-instruct"
"https://huggingface.co/nvidia/Llama-3.1-Nemotron-8B-UltraLong-4M-Instruct" ("https://huggingface.co/nvidia/Llama-3.1-Nemotron-8B-UltraLong-2M-Instruct", "https://huggingface.co/nvidia/Llama-3.1-Nemotron-8B-UltraLong-1M-Instruct")
"https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct"
"https://huggingface.co/google/gemma-3-4b-it-qat-q4_0-gguf" ("https://huggingface.co/google/gemma-3-4b-it")

"small medium models"

"https://huggingface.co/Qwen/Qwen2.5-14B-Instruct-1M"
"https://huggingface.co/google/gemma-3-12b-it-qat-q4_0-gguf" ("https://huggingface.co/google/gemma-3-12b-it")

"Medium models"
"https://huggingface.co/nvidia/Llama-3.1-Nemotron-70B-Instruct-HF" ("https://huggingface.co/nvidia/Llama-3_1-Nemotron-51B-Instruct")
"https://huggingface.co/Qwen/Qwen2.5-72B-Instruct-GPTQ-Int4"
"https://huggingface.co/google/gemma-3-27b-it-qat-q4_0-gguf" ("https://huggingface.co/google/gemma-3-27b-it")
"BIG MODELS"
"https://huggingface.co/nvidia/Llama-3_1-Nemotron-Ultra-253B-v1"

In [ ]:
! pip -q install datasets
! pip install wget
! pip install rake-nltk
! pip install RapidFuzz
! pip install spacy






In [15]:
import json
import datasets
import wget

Let's first download the recipes dataset. After that, we can load the dataset via the huggingface `datasets` library, which offers easy integration with `transformers`.

In [ ]:
wget.download("https://people.cs.kuleuven.be/~thomas.bauwens/irse_documents_2025_recipes.parquet")

In [17]:
dataset = datasets.load_dataset("parquet", data_files="./irse_documents_2025_recipes.parquet")['train']

Generating train split: 231637 examples [00:00, 709124.07 examples/s]


`datasets` allows us to index directly into a `Dataset` object and easily access the data associated with a sample. You can find more information about working with datasets [here](https://huggingface.co/docs/datasets/access).

In [18]:
print("One document:")
for example in dataset:
    for k,v in example.items():
        print(f"'{k}' = {v}\n")
    break

One document:
'name' = arriba baked winter squash mexican style

'ingredients' = winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt

'steps' = make a choice and proceed with recipe, depending on size of squash , cut into half or fourths, remove seeds, for spicy squash , drizzle olive oil or melted butter over each cut squash piece, season with mexican seasoning mix ii, for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece, season with sweet mexican spice mix, bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin, be careful not to burn the squash especially if you opt to use sugar or butter, if you feel more comfortable , cover the squash with aluminum foil the first half hour , give or take , of baking, if desired , season with salt

'tags' = 60-minutes-or-less, time-to-make, course, main-ingredient, cuisine, preparation, occasion, north-american, side-

We can also load the `queries.json` file, which contains the gold queries created by the instructors. You can use this to debug your retriever and estimate MAP (see project instructions for details).

In [19]:
wget.download("https://people.cs.kuleuven.be/~thomas.bauwens/irse_queries_2025_recipes.json")
queries = json.load(open("./irse_queries_2025_recipes.json", "r"))

In [21]:
print(queries["queries"][0])

{'q': 'What temperature should I pre-heat my oven to when making chicken quesadillas?', 'r': [[167945, 1], [167954, 1], [21548, 1], [218187, 1], [168524, 1], [68174, 1], [34390, 1], [34410, 1], [85623, 1], [46749, 1], [83613, 1], [210101, 1], [192707, 1], [19157, 1], [46809, 1], [168697, 1], [139022, 1], [168732, 1], [151851, 1], [45356, 1], [45357, 1], [40241, 1], [6453, 1], [179511, 1], [168270, 1], [19788, 1], [223067, 1], [19339, 1], [98716, 1], [191431, 1], [25072, 1]], 'a': '375 is a good temperature, but can go as low as 350 or as high as 400. Adjust times accordingly (longer for lower temperatures).'}


You can see that the `queries` dictionary object contains a list of dictionaries, consisting of query (`q`), answer (`a`), and relevant documents (`r`) fields. The integer values in `r` correspond to the `official_id` field in the `recipes.parquet` dataset (see above), along with a relevance score.



Now that the dataset and queries have been loaded, you are free to implement your RAG pipeline. You are welcome to use any implementation of TF-IDF that you are familiar with. Keep in mind that the TF-IDF model must be fit on the recipes dataset provided above, although you are free to experiment what field is most salient for your relevant document search.

In [ ]:
# TODO: implement TF-IDF
# TODO: fit TF-IDF model on recipes dataset
# TODO: implement nearest neighbors search


import numpy as np
import pandas as pd
import json
import os
from collections import Counter, defaultdict

from rake_nltk import Rake
from rapidfuzz import fuzz
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from spacy.matcher import PhraseMatcher
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

import spacy
import re

import joblib
from scipy.sparse import save_npz, load_npz

SAMPLE_SIZE = 200
CLUSTER_N = 25
TOP_N = 10
VERSION = 0.0
OUTPUT_MAP_PATH = f'storage/normalization_maps/map_{VERSION}.json'
OUTPUT_VECTOR_PATH = f'storage/vectorizers/vector_{VERSION}.joblib'
OUTPUT_MATRIX_PATH = f'storage/matrixs/matrix_{VERSION}.npz'
OUTPUT_DENSE_MATRIX_PATH = f'storage/dense_matrixs/dense_matrix_{VERSION}.npz'
OUTPUT_LSA_PIPELINE_PATH = f'storage/lsa_pipeline/lsa_pipeline_{VERSION}.joblib'


SEPARATE_MODE = "top"
SEPARATE_PARAM = 3
MODE = "derivative"
PARAM = "auto"
ALPHA = 0.5
MIN_SCORE = 0.5

class TFIDF():
    
    
    
    
    
    def __init__(self, df: pd.DataFrame):
        digits = re.compile(r'.*\d+.*')
        nlp = spacy.load("en_core_web_sm")
        df = df.copy()

        if not os.path.exists(os.path.dirname( OUTPUT_MAP_PATH)):
            os.mkdir(os.path.dirname( OUTPUT_MAP_PATH))

        if not os.path.exists(os.path.dirname( OUTPUT_VECTOR_PATH)):
            os.mkdir(os.path.dirname( OUTPUT_VECTOR_PATH))

        if not os.path.exists(os.path.dirname( OUTPUT_MATRIX_PATH)):
            os.mkdir(os.path.dirname( OUTPUT_MATRIX_PATH))


    def TFIDF(self):

        if os.path.exists(OUTPUT_MAP_PATH) and  os.path.exists(OUTPUT_VECTOR_PATH) and  os.path.exists(OUTPUT_MATRIX_PATH) \
              and os.path.exists(OUTPUT_DENSE_MATRIX_PATH) and os.path.exists(OUTPUT_LSA_PIPELINE_PATH):
            with open(OUTPUT_MAP_PATH, "r") as f:
                normalization_map = json.load(f)

     
            vectorizer = joblib.load(OUTPUT_VECTOR_PATH)

            tfidf_matrix = load_npz(OUTPUT_MATRIX_PATH)

            lsa_pipeline = joblib.load(OUTPUT_LSA_PIPELINE_PATH)

            data = np.load(OUTPUT_DENSE_MATRIX_PATH)
            dense_matrix = data["dense_matrix"]
            
            return normalization_map, vectorizer, tfidf_matrix, lsa_pipeline, dense_matrix
        else:
            
            self._apply_lemmatization()
            phrases = self._extract_phrases(self.df['lemmatized'])
            grouped_tags  = self._cluster_phrases(phrases, n_clusters=CLUSTER_N)
            normalization_map = self._build_canonical_mapping(grouped_tags)
            self._tokenise(normalization_map)
            matrix, vectorizer = self._vectorize_texts(self.df['normalized_text'])

            lsa_pipeline, dense_matrix = self._compute_dense_representation(matrix)

            return normalization_map, vectorizer, matrix, lsa_pipeline, dense_matrix, self.df.copy()
            
            
    def _compute_dense_representation(self, tfidf_matrix):
        svd = TruncatedSVD(n_components=100, random_state=42)
        normalizer = Normalizer(copy=False)
        lsa_pipeline = make_pipeline(svd, normalizer)
        dense_matrix = lsa_pipeline.fit_transform(tfidf_matrix)
        np.savez_compressed(OUTPUT_DENSE_MATRIX_PATH, dense_matrix=dense_matrix)
        joblib.dump(lsa_pipeline, OUTPUT_LSA_PIPELINE_PATH)
        return lsa_pipeline, dense_matrix

    def _prepare_relevant_text(self):
        # adjust columns as needed
        self.df['text'] = self.df[['name', 'tags', 'ingredients', 'description']].astype(str).agg(' '.join, axis=1)

    def lemmatize_text(self,text):
        doc = self.nlp(text.lower())
        return ' '.join([tok.lemma_ for tok in doc if not tok.is_punct and not tok.is_space and not self.digits.match(tok.lemma_)])
    
    def _apply_lemmatization(self):
        self.df['lemmatized'] = self.df['text'].apply(self._lemmatize_text)
    
    
    def _extract_phrases(self, texts, sample_size=SAMPLE_SIZE):
        if sample_size == -1:
            sample_size = len(texts)
        rake = Rake()
        phrases = []
        for txt in texts.sample(min(sample_size, len(texts)), random_state=16):
            rake.extract_keywords_from_text(txt)
            phrases.extend(rake.get_ranked_phrases())
        return phrases
    
    def _cluster_phrases(self, phrases, n_clusters=20):
        counts = Counter(phrases)
        filtered = [p for p, c in counts.items() if 3 < len(p) < 50 and c > 2]
        phrase_list = list(filtered)

        # build distance matrix
        dist = [
            [100 - fuzz.token_sort_ratio(a, b) for b in phrase_list]
            for a in phrase_list
        ]
        clustering = AgglomerativeClustering(
            n_clusters=n_clusters,
            affinity='precomputed',
            linkage='average' # minimizes variance
        )
        """
        Distance Matrix vs. Raw Vectors
            We’re clustering on a precomputed distance matrix (100 – fuzz.token_sort_ratio), which is not guaranteed to satisfy the Euclidean metric assumptions that Ward linkage requires.

            Ward’s Requirement
            The Ward method minimizes total within‐cluster variance and assumes you’re clustering raw feature vectors in a Euclidean space. It uses the notion of “cluster centroids” and squared‐Euclidean distances to decide merges.

            Average (UPGMA) Flexibility
            Average linkage (a.k.a UPGMA) simply computes the average pairwise distance between all members of two clusters. It works fine with any symmetric distance matrix, even if it isn’t Euclidean. That makes it a safer, more general choice when your “points” are really just fuzzy‐distance scores.

        """
        labels = clustering.fit_predict(dist) 
        """
        With a defaultdict(list), you can do: grouped[label].append(phrase) without first checking if label not in grouped: grouped[label] = [].
        """

        grouped = defaultdict(list)
        for i, lbl in enumerate(labels):
            grouped[f'group_{lbl}'].append(phrase_list[i])
        return grouped
        
    def _build_canonical_mapping(self,grouped_tags):
        mapping = {}
        for variants in grouped_tags.values():
            # pick the shortest, most central variant as canonical
            canonical = min(variants, key=lambda x: (len(x), variants.index(x)))
            for v in variants:
                mapping[v.lower()] = canonical.lower()
                
        with open(OUTPUT_MAP_PATH, "w") as f:
            json.dump(mapping, f, indent=2)
            print(f"Saved normalization map → {OUTPUT_MAP_PATH}")
        return mapping
    
    def normalize_text(sefl,text, matcher, mapping):
        doc = sefl.nlp(text)
        norm = text.lower()
        for _, start, end in matcher(doc):
            span = doc[start:end].text.lower()
            tag  = mapping.get(span)
            if tag:
                norm = norm.replace(span, tag)
        return norm

    def _tokenise(self, mapping):
        matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        patterns = [self.nlp.make_doc(phrase) for phrase in mapping.keys()]
        matcher.add("TAGS", patterns)
        self.df['normalized_text'] = self.df['lemmatized'].apply(
        lambda txt: self._normalize_text(txt, matcher, mapping)
            )

    def _vectorize_texts(self,texts):
        vect = TfidfVectorizer()
        mat  = vect.fit_transform(texts)
        joblib.dump(vect, OUTPUT_VECTOR_PATH)
        save_npz(OUTPUT_MATRIX_PATH, mat)
        print(f"Saved TF-IDF vectorizer → {OUTPUT_VECTOR_PATH}")
        print(f"Saved TF-IDF matrix     → {OUTPUT_MATRIX_PATH}")
        return mat, vect


class Ranking():

    def __init__(self, df_raw: pd.DataFrame):

        nlp = spacy.load("en_core_web_sm")
        matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

        tfidf = TFIDF(df_raw)
        normalization_map , vectorizer , tfidf_matrix, lsa_pipeline, dense_matrix, df = tfidf.TFIDF()

        patterns = [nlp.make_doc(phrase) for phrase in normalization_map.keys()]
        matcher.add("TAGS", patterns)


    def _similarities_sparse(self, query):
        query_sparse = query.copy()
        query_sparse = self._preprocess_query(query_sparse)
        vec_sparse = self.vectorizer.transform([query_sparse])
        sims = cosine_similarity(vec_sparse, self.tfidf_matrix).flatten()
        return sims
    
    def _similarities_dense(self, query, metric):
        query_dense = query.copy()
        q_vec = self.lsa_pipeline.transform([query_dense])

        sims = cosine_similarity(q_vec, self.dense_matrix).flatten()
        return sims

        
    def compute_jaccard_score(doc_tokens: set, query_tokens: set) -> float:
  
        if not query_tokens:
            return 0.0
        inter = doc_tokens & query_tokens
        union = doc_tokens | query_tokens
        return len(inter) / len(union) if union else 0.0
        
    def query(self, query: str, seperate_mode:str, seperate_param,  mode:str, param, alpha: float, min_score:int ):
        """
        Dynamic search based on cosine similarity with multiple cutoff strategies.
        
        Args:
            query: User query string.
            mode: One of "relative", "statistical", "absolute", or "derivative".
            param: Sensitivity parameter for the mode.
                relative: >= max * param
                statistical: >= mean + param *std
                absolute: >= param
                dervivative: find first occurance of diff > threshold or "auto" -> find max derivative absolute value
            min_results: Minimum number of results to return.

        Returns:
            DataFrame with matched recipes and scores.
        """

        assert(0 < alpha < 1)
        sims_sparse = self._similarities_sparse(query)
        sims_dense = self._similarities_dense(query)

        if seperate_param:
            sparse_clear_idx = self._rank(sims_sparse, seperate_mode,seperate_param, min_score)
            dense_clear_idx = self._rank(sims_dense, seperate_mode,seperate_param, min_score)

        sims = alpha*sims_sparse * (1- alpha) * sims_dense

        spare_dense_idx = self._rank(sims, mode, param)

        combined = sparse_clear_idx | dense_clear_idx | spare_dense_idx


        self.df['normalized_text']
        # Retrieve results
        res = self.df.iloc[combined].copy()

        res["score_sparse"] = sims_sparse[combined]
        res["score_dense"] = sims_dense[combined]
        return res.sort_values("score", ascending=False)

    def _rank(self, sims, mode, param, min_score):
        if sims.size == 0:
            return None
        if mode == "top":
            thr = min(param, sims.size)
            sorted_ix = sims.argsort()[::-1]
            sel = sorted_ix[:thr]
            idx = sel[sims[sel] >= min_score]
        if mode == "relative":
            thr = sims.max() * param
            idx = np.where(sims >= thr)[0]
        elif mode == "statistical":
            thr = sims.mean() + param * sims.std()
            idx = np.where(sims >= thr)[0]
        elif mode == "absolute":
            idx = np.where(sims >= param)[0]
        elif mode == "derivative":
            idx = self._select_top_by_derivative(sims, threshold=param)
        else:
            raise ValueError(f"Unknown mode: {mode}")
        
        return idx

    def _select_top_by_derivative(scores: np.ndarray, threshold="auto") -> np.ndarray:
        sorted_ix = scores.argsort()[::-1]
        sorted_scores = scores[sorted_ix]
        diffs = np.abs(np.diff(sorted_scores))

        if diffs.size == 0:
            cutoff = sorted_scores.size
        else:
            if threshold == "auto":
                drop_ix = np.argmax(diffs)
                cutoff = drop_ix + 1
            else:
                large_drops = np.where(diffs > threshold)[0]
                cutoff = (large_drops[0] + 1) if large_drops.size > 0 else sorted_scores.size

        sel = sorted_ix[:cutoff]
        return sel
        
    
    def _preprocess_query(self,query: str) -> str:
        matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        patterns = [self.nlp.make_doc(phrase) for phrase in self.normalization_map.keys()]
        matcher.add("TAGS", patterns)
        lem = self.tfidf.lemmatize_text(query)
        return self.tfidf.normalize_text(lem, matcher, self.normalization_map)
    
class Evaluate():
    
    def __init__():
        pass

def dcg(scores):

    return np.sum([
        (2**rel - 1) / np.log2(rank + 2)  # rank starts at 0
        for rank, rel in enumerate(scores)
    ])

def ndcg(relevance_scores, ideal_scores):
    dcg_val = dcg(relevance_scores)
    idcg_val = dcg(ideal_scores)
    return dcg_val / idcg_val if idcg_val > 0 else 0.0

def evaluate_with_relevance(df_queries, search_fn, top_k=10, threshold=0.5):
    macro_ndcg_scores = []
    macro_precisions = []
    macro_recalls = []
    macro_f1s = []

    all_pred_bin = []
    all_true_bin = []

    df_queries = df_queries["queries"]

    ranking = Ranking()
    print("Done with TFIDF computation")

    for row in df_queries:
        query = row["q"]
        relevant_ids = [entry[0] for entry in row["r"] ] 
        relevance_values predicted_scores= [entry[1] for entry in row["r"] ] 
 


        predicted_doc = ranking.query(query, SEPARATE_MODE, 
                                        SEPARATE_PARAM,
                                        MODE,
                                        PARAM,
                                        ALPHA,
                                        MIN_SCORE)
        
        predicted_ids = [entry["official_id"] for entry in predicted_doc]
        # Get predicted document ids from system

        # Build relevance vector for predicted docs
        predicted_scores = [rel_dict.get(doc_id, 0.0) for doc_id in predicted_ids]

        # Ideal sorted relevance scores
        ideal_scores = sorted(rel_dict.values(), reverse=True)[:top_k]

        # nDCG for this query
        macro_ndcg_scores.append(ndcg(predicted_scores, ideal_scores))

        # Binary relevance for metrics
        predicted_set = set(predicted_ids)
        true_positives = len(predicted_set & relevant_ids)

        precision = true_positives / len(predicted_set) if predicted_set else 0
        recall = true_positives / len(relevant_ids) if relevant_ids else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        macro_precisions.append(precision)
        macro_recalls.append(recall)
        macro_f1s.append(f1)

        for pid in predicted_ids:
            all_pred_bin.append(pid in relevant_ids)
        for rid in relevant_ids:
            all_true_bin.append(rid in predicted_set)

    macro = {
        "ndcg": np.mean(macro_ndcg_scores),
        "precision": np.mean(macro_precisions),
        "recall": np.mean(macro_recalls),
        "f1": np.mean(macro_f1s),
    }

    # Micro
    micro_precision = sum(all_pred_bin) / len(all_pred_bin) if all_pred_bin else 0
    micro_recall = sum(all_pred_bin) / len(all_true_bin) if all_true_bin else 0
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0

    micro = {
        "precision": micro_precision,
        "recall": micro_recall,
        "f1": micro_f1
    }

    return macro, micro


    



You can see that the `queries` dictionary object contains a list of dictionaries, consisting of query (`q`), answer (`a`), and relevant documents (`r`) fields. The integer values in `r` correspond to the `official_id` field in the `recipes.parquet` dataset (see above), along with a relevance score.



Once your TF-IDF model has been implemented and fit on the recipes dataset, you can experiment with retrieving the k-most relevant documents for the queries provided below:

In [ ]:
sample_queries = [
    "a cajun style gumbo with an easy roux",
    "I am feeling like eating shrimp tacos tonight. What's a good recipe?",
    "recipe for easy vegetarian lasagna",
    "How do I make spageti and meatballs?",
    "15 minute lunch recipe",
    "Give me suggestion for some easy vegetarian weeknight dinner recipes"
]

For a given query and set of relevant documents, you are also required to create a prompt that instructs a model to complete a certain task (e.g. recipe recommendation). You should experiment with formatting the prompt, as language models have been shown to be sensitive to the exact verbiage of instructions.

In [ ]:
prompt = f"""

YOUR PROMPT GOES HERE

"""

In [ ]:
irrelevant_context = """
Richard Gary Brautigan (January 30, 1935 – c. September 16, 1984)
was an American novelist, poet, and short story writer. A prolific writer,
he wrote throughout his life and published ten novels, two collections of
short stories, and four books of poetry. Brautigan's work has been published
both in the United States and internationally throughout Europe, Japan,
and China. He is best known for his novels Trout Fishing in America (1967),
In Watermelon Sugar (1968), and The Abortion: An Historical Romance 1966 (1971).
"""

Before loading a model from the HuggingFace hub, you will likely want to create an account at https://huggingface.co/ so that you can get an [**access token**](https://huggingface.co/docs/hub/security-tokens) for models which require identification before usage.

- On your personal machine, you can input this access token by running `huggingface-cli login` in a terminal window.

- In Colab, click the icon in the left sidebar that looks like a key, and *Add a secret* called `HF_TOKEN`.

If you don't do this, you risk running into a "Cannot access gated repo" error.

In [ ]:
from google.colab import userdata
userdata.get("HF_TOKEN")

**IMPORTANT**: only run the following code when you have implemented a working retrieval system. When you are ready to work with language models, navigate to the menu bar in Colab and select **Runtime > Change runtime type > T4 GPU**. If you find yourself working on not GPU-intenstive tasks in this notebook, change your runtime back to CPU to preserve access.


In [ ]:
! pip -q install git+https://github.com/huggingface/transformers

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
! pip -q install datasets bitsandbytes accelerate xformers einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 866.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import transformers
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM

The code below will load a Mistral 7B instruct model and quantize it via `bitesandbytes`. Doing so will ensure that the model will not take up too much memory and make inference more efficient. Note that the call to `AutoModelForCausalLM.from_pretrained()` will take a while, as the model's weights must be downloaded from the huggingface hub. Also note that you are not restricted to using Mistral, and are welcome to experiment with other models (though you will have more luck with chat and instruction-tuned variants).

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"
bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto'
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

A tokenizer is required in order to convert strings into integer sequences that can be passed as input to the model.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
input_string = prompt + sample_queries[0]

In [ ]:
encoded_prompt = tokenizer(input_string, return_tensors="pt", add_special_tokens=False)
encoded_prompt = encoded_prompt.to("cuda")

This is the final generation step, where a forward pass must be made through the entire model. Since the model is large (even after quantization), it might take a while.

In [ ]:
generated_ids = model.generate(**encoded_prompt, max_new_tokens=1000, do_sample=True)
decoded = tokenizer.batch_decode(generated_ids)
print(decoded[0])

We can see that, even without additional context and reference documents, the model is able to generate very coherent recipe instructions. Now, it is up to you to experiment with the RAG framework and see if you can further improve the quality of the model's generation with relevant documents. Refer to the assignment handout for the exact questions we expect you to answer.  